In [2]:
import numpy as np

# ==========================================
# 第一步：把大矩陣「切塊」，定義出我們的 6 個零件
# (這裡的數字完全照抄你投影片 Example 裡的矩陣)
# ==========================================

# 1. 舊重心 (Means)
mu1 = np.array([[1], 
                [-1]])  # 我們想預測的 (X1, X3)
mu2 = np.array([[0]])   # 我們已知的 (X2)

# 2. 變異數與共變異數 (Covariances)
Sigma11 = np.array([[2, 0], 
                    [0, 2]]) # 想預測的變異數

Sigma22 = np.array([[3]])    # 已知的變異數

Sigma12 = np.array([[1], 
                    [1]])    # 兩者的關聯性

Sigma21 = np.array([[1, 1]]) # 關聯性的轉置 (橫的)


# ==========================================
# 第二步：帶入新情報 (Observation)
# ==========================================
# 假設我們今天真的觀察到 X2 的數值是 6
x2_observed = np.array([[6]])


# ==========================================
# 第三步：施展公式魔法！
# 秘訣：在 numpy 裡，矩陣相乘使用 @ 符號，反矩陣使用 np.linalg.inv()
# ==========================================

# 1. 計算新重心 (Conditional Mean)
# 公式： mu1 + Sigma12 * (Sigma22 的反矩陣) * (實際觀察值 - mu2)
new_mu = mu1 + Sigma12 @ np.linalg.inv(Sigma22) @ (x2_observed - mu2)

# 2. 計算新變異數 (Conditional Covariance)
# 公式： Sigma11 - Sigma12 * (Sigma22 的反矩陣) * Sigma21
new_Sigma = Sigma11 - Sigma12 @ np.linalg.inv(Sigma22) @ Sigma21

# ==========================================
# 第四步：印出結果，見證奇蹟
# ==========================================
print("=== 預測結果 ===")
print(f"當觀察到 X2 = {x2_observed[0][0]} 時：\n")

print("👉 新的重心位置 (Expected Value):")
print(new_mu)
# 如果把 x2=6 代入手算的公式： 1 + (1/3)*6 = 3, -1 + (1/3)*6 = 1
# 你會發現程式印出來剛好就是 [[3], [1]]！

print("\n👉 新的變異數 (Covariance Matrix):")
print(new_Sigma)
# 你會看到印出來就是 [[1.6666, -0.3333], [-0.3333, 1.6666]]
# 完美對應投影片最右下角的 5/3 與 -1/3！

=== 預測結果 ===
當觀察到 X2 = 6 時：

👉 新的重心位置 (Expected Value):
[[3.]
 [1.]]

👉 新的變異數 (Covariance Matrix):
[[ 1.66666667 -0.33333333]
 [-0.33333333  1.66666667]]


In [3]:
import numpy as np
import scipy.stats as stats

# ==========================================
# 步驟 1：製造「現實世界」的樣本 (藍色階梯)
# ==========================================
n = 20  # 假設我們只有 20 個樣本
# 隨機產生 20 個數據 (我們偷偷讓它符合常態分佈)
y_sample = np.random.normal(loc=0, scale=1, size=n)

# 數學公式第一步：要把數字從小到大排好
y_sorted = np.sort(y_sample)

# 計算 Empirical CDF (藍色階梯的累積高度)
# np.arange(1, n+1) 會產生 [1, 2, ..., 20]，除以 n 就是 [0.05, 0.10, ..., 1.0]
ecdf = np.arange(1, n + 1) / n 

# ==========================================
# 步驟 2：召喚「神界」的完美常態分佈 (紅色曲線)
# ==========================================
# 使用 stats.norm.cdf 算出這些點在完美常態分佈下的理論累積機率
theoretical_cdf = stats.norm.cdf(y_sorted)

# ==========================================
# 步驟 3：算最大落差 (公式裡的 sup 與絕對值)
# ==========================================
# 把藍色高度減掉紅色高度，取絕對值 np.abs()，然後找出最大值 np.max()
Dn = np.max(np.abs(ecdf - theoretical_cdf))

print(f"我們手動算出來的最大落差 D_n 是: {Dn:.4f}")

# ------------------------------------------------
# 業界作弊法：其實上面這一大串，scipy 早就寫好了！
# 只要一行，它會吐出 D 值跟 p-value
# ------------------------------------------------
statistic, p_value = stats.kstest(y_sample, 'norm')
print(f"業界套件直接算的 D 值是: {statistic:.4f}")

我們手動算出來的最大落差 D_n 是: 0.1764
業界套件直接算的 D 值是: 0.1764


In [4]:
import scipy.stats as stats

print("=== 測試投影片 Slide 16 的反向查表 ===")

# 投影片裡的 p 值：0.025, 0.16, 0.5, 0.84, 0.975
p_values = [0.025, 0.16, 0.5, 0.84, 0.975]

for p in p_values:
    # 呼叫常態分佈的反函數 (ppf)
    # 邏輯：請告訴我，累積了 p 的機率時，x 的門檻是多少？
    x = stats.norm.ppf(p)
    
    # 印出來對答案 (投影片寫大約 -2, -1, 0, 1, 2)
    print(f"當 p = {p:<5} 時，對應的門檻分數 x = {x:.2f}")

# ------------------------------------------------
# 補充：如果我是有一批「真實資料 (y_sample)」，我要怎麼找分位數？
# ------------------------------------------------
# 這時候就輪到 numpy 出馬了！
# 假設我要找剛剛那 20 個學生的「中位數 (50%)」跟「前 97.5% 的門檻」
q_50 = np.quantile(y_sorted, 0.5)
q_975 = np.quantile(y_sorted, 0.975)

print("\n=== 真實資料的 Quantile ===")
print(f"這批資料的 50% 門檻 (中位數): {q_50:.2f}")
print(f"這批資料的 97.5% 門檻: {q_975:.2f}")

=== 測試投影片 Slide 16 的反向查表 ===
當 p = 0.025 時，對應的門檻分數 x = -1.96
當 p = 0.16  時，對應的門檻分數 x = -0.99
當 p = 0.5   時，對應的門檻分數 x = 0.00
當 p = 0.84  時，對應的門檻分數 x = 0.99
當 p = 0.975 時，對應的門檻分數 x = 1.96

=== 真實資料的 Quantile ===
這批資料的 50% 門檻 (中位數): -0.32
這批資料的 97.5% 門檻: 1.10
